In [1]:
import os
import re
import json
import glob
import asyncio
import requests
from pathlib import Path
from typing import TypedDict, List, Dict, Any, Optional, Iterable
from datetime import datetime
from collections import defaultdict

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=False)

import chromadb  # <-- pure chroma
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama  # hanya LLM-nya aja

# MCP client
from fastmcp import Client
from fastmcp.client.transports import StreamableHttpTransport

import pandas as pd

# =========================
# === CONFIG (ENV) ========
# =========================
# ---- Datahub (Sparse Retrieval) ----
DATAHUB_GMS_URL   = os.getenv("DATAHUB_GMS_URL", "http://datahub-gms:8080")
DATAHUB_GRAPHQL   = os.getenv("DATAHUB_GRAPHQL_URL", f"{DATAHUB_GMS_URL}/api/graphql")
DATAHUB_TOKEN     = os.getenv("DATAHUB_TOKEN")
DATAHUB_ACTOR     = os.getenv("DATAHUB_ACTOR")
DATAHUB_TIMEOUT   = int(os.getenv("DATAHUB_TIMEOUT", "15"))
DATAHUB_MAX_HITS  = int(os.getenv("DATAHUB_MAX_HITS", "10"))

# ---- RAG ----
JSON_INPUT       = os.getenv("EXPORT_JSONL_OUTPUT", "./datahub_metadata/*.jsonl").strip()
CHROMA_DIR       = os.getenv("CHROMA_DIR", "./chroma_db")
COLLECTION_NAME  = os.getenv("COLLECTION_NAME", "classicmodels")
EMBED_MODEL      = os.getenv("EMBED_MODEL", "bge-m3:latest")
OLLAMA_BASE_URL  = os.getenv("OLLAMA_URL", "http://localhost:11434")
CHAT_MODEL       = os.getenv("CHAT_MODEL", "HridaAI/hrida-t2sql:q8_0")
TOP_K            = int(os.getenv("TOP_K", "5"))

# ---- Probe controls ----
USE_DYNAMIC_PROBE = os.getenv("USE_DYNAMIC_PROBE", "1").strip() == "1"

# Legacy probe (fallback)
RETRIEVAL_PROBE = (
    " measure measures metric metrics sum total amount price revenue sales gross net "
    "quantity qty count average avg mean distinct group by bucket date time timestamp "
    "created updated order_date month week quarter year category region customer status id"
)

MICRO_PROBE = ["id", "date", "status"]

# ---- Rerank weights ----
SPARSE_WEIGHT  = float(os.getenv("SPARSE_WEIGHT", "1.0"))
SPARSE_BONUS   = float(os.getenv("SPARSE_BONUS", "0.75"))
INTENT_BONUS   = float(os.getenv("INTENT_BONUS", "0.35"))
RRF_K          = int(os.getenv("RRF_K", "60"))

# ---- MCP ----
MCP_BASE    = os.getenv("MCP_BASE", "http://mcp_server:8000/mcp")
MCP_TIMEOUT = int(os.getenv("MCP_CLIENT_TIMEOUT", "60"))

# ---- Debug flag (0=off, 1=on) ----
SQL_DEBUG = int(os.getenv("SQL_DEBUG", "1"))
def _dbg_print_two(label: str, sql: str):
    if SQL_DEBUG:
        print(f"\n{label}")
        print(sql)

# =========================
# === RAG (STATE) =========
# =========================
class RAGState(TypedDict):
    jsonl_paths: List[str]
    question: str
    added: int
    updated: int
    skipped: int
    context: str
    answer: str
    draft_sql: str
    final_sql: str

# =========================
# === UTIL ================
# =========================
def resolve_jsonl_paths(spec: str) -> List[str]:
    paths: List[str] = []
    if not spec:
        return paths
    p = Path(spec)
    if any(ch in spec for ch in "*?[]"):
        paths.extend(glob.glob(spec))
    elif p.is_dir():
        paths.extend(str(x) for x in p.glob("*.jsonl"))
    elif p.is_file():
        paths.append(str(p))
    else:
        paths.extend(glob.glob(spec))
    out, seen = [], set()
    for x in paths:
        xp = str(Path(x))
        if xp.lower().endswith(".jsonl") and xp not in seen:
            seen.add(xp)
            out.append(xp)
    return out

def build_vector_store():
    """
    Pure chroma, no langchain wrapper.
    """
    os.makedirs(CHROMA_DIR, exist_ok=True)
    client = chromadb.PersistentClient(path=CHROMA_DIR)
    col = client.get_or_create_collection(COLLECTION_NAME)
    return col

def read_jsonl(path: str) -> Iterable[tuple[str, Optional[Dict[str, Any]]]]:
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if not s:
                continue
            try:
                obj = json.loads(s)
            except Exception:
                obj = None
            yield s, obj

def _safe_iso_to_dt(s: Optional[str]) -> Optional[datetime]:
    if not s:
        return None
    try:
        return datetime.fromisoformat(s)
    except Exception:
        return None

def _preferred_id(meta: dict, source_name: str) -> str:
    if meta.get("field_urn"):
        return meta["field_urn"]
    if meta.get("urn"):
        return meta["urn"]
    raise ValueError(f"{source_name}: JSONL row does not have field_urn or urn")

# =========================
# === SQL-AWARE CARDS =====
# =========================
def _column_card(meta: Dict[str, Any]) -> str:
    db = meta.get("database", "") or meta.get("db", "")
    sch = meta.get("schema", "") or meta.get("schemaName", "")
    tbl = meta.get("table", "") or meta.get("tableName", "")
    col = meta.get("column", "") or meta.get("columnName", "")
    dtype = meta.get("dtype") or meta.get("dataType") or meta.get("type", "")
    desc = meta.get("description") or meta.get("brief", "") or ""
    aliases = meta.get("aliases") or meta.get("synonyms") or meta.get("aka") or []
    if isinstance(aliases, str):
        aliases = [aliases]
    sample_vals = meta.get("sample_values") or meta.get("samples") or meta.get("examples") or []
    if isinstance(sample_vals, dict):
        sample_vals = list(sample_vals.values())
    sample_txt = ", ".join(map(lambda x: str(x)[:40], sample_vals[:5]))

    fqn = ".".join([p for p in [db, sch, tbl, col] if p])
    return (
        f"TABLE: {tbl}\n"
        f"COLUMN: {col}\n"
        f"TYPE: {dtype}\n"
        f"ALIASES: {', '.join(map(str, aliases))}\n"
        f"DESC: {desc}\n"
        f"SAMPLE: {sample_txt}\n"
        f"FQN: {fqn}"
    ).strip()

# =========================
# === SPARSE (DATAHUB) ====
# =========================
def datahub_sparse_search(q: str, count: int = None) -> list[str]:
    if not DATAHUB_GRAPHQL or not DATAHUB_TOKEN:
        return []
    c = count or DATAHUB_MAX_HITS

    gql = """
    query searchDatasets($q: String!, $count: Int!) {
      search(input: {
        type: DATASET,
        query: $q,
        start: 0,
        count: $count
      }) {
        searchResults {
          entity {
            urn
          }
        }
      }
    }
    """.strip()

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {DATAHUB_TOKEN}",
    }
    if DATAHUB_ACTOR:
        headers["X-DataHub-Actor"] = DATAHUB_ACTOR

    try:
        resp = requests.post(
            DATAHUB_GRAPHQL,
            json={"query": gql, "variables": {"q": q, "count": c}},
            headers=headers,
            timeout=DATAHUB_TIMEOUT,
        )
        resp.raise_for_status()
        data = resp.json()
        hits = (data.get("data") or {}).get("search") or {}
        out: list[str] = []
        for h in hits.get("searchResults") or []:
            ent = h.get("entity") or {}
            urn = ent.get("urn")
            if urn:
                out.append(urn)
        return out
    except Exception as e:
        print(f"[datahub_sparse_search] error: {e}")
        return []

def _dedup_keep_order(tokens: List[str]) -> List[str]:
    return list(dict.fromkeys([t for t in tokens if t and isinstance(t, str)]))

def build_dynamic_probe(question: str, max_tokens: int = 28) -> str:
    q = (question or "").lower()

    # Query identitas murni (nama) → jangan pakai MICRO_PROBE
    identity_only = (
        any(k in q for k in ["name", "nama"])
        and not any(k in q for k in [
            "sum", "total", "jumlah", "revenue", "sales", "omset", "gross", "net",
            "count", "berapa banyak", "how many", "hitung",
            "avg", "average", "mean", "rata", "rata-rata",
            "today", "yesterday", "last", "this", "bulan", "minggu", "tahun", "quarter",
            "month", "week", "year", "202", "201", "tanggal", "hari", "mingguan", "bulanan"
        ])
    )

    base: List[str] = [] if identity_only else MICRO_PROBE[:]  # ["id", "date", "status"]

    if any(k in q for k in ["avg", "average", "mean", "rata", "rata-rata"]):
        base += ["avg", "mean", "average", "metric", "measure", "price"]
    if any(k in q for k in ["sum", "total", "jumlah", "revenue", "sales", "omset", "gross", "net"]):
        base += ["sum", "total", "amount", "revenue", "sales", "price", "gross", "net"]
    if any(k in q for k in ["count", "berapa banyak", "how many", "hitung"]):
        base += ["count", "qty", "quantity", "number"]

    if any(k in q for k in [
        "today", "yesterday", "last", "this", "bulan", "minggu", "tahun", "quarter",
        "month", "week", "year", "quarter", "202", "201", "tanggal", "hari", "mingguan", "bulanan"
    ]):
        base += ["date", "timestamp", "month", "week", "quarter", "year", "created", "updated", "order_date"]

    # fokus customer
    if any(k in q for k in ["customer", "customers", "pelanggan"]):
        base += ["customer", "customer name"]

    # lokasi hanya kalau disebut eksplisit
    if any(k in q for k in ["country", "negara", "city", "kota", "region", "wilayah"]):
        base += ["country", "city", "region"]

    # kalau nanya "name / nama", fokus ke customer name
    if any(k in q for k in ["name", "nama"]):
        base += ["name", "customer name"]

    # khusus kalau user eksplisit nyebut first / last name
    if any(k in q for k in ["first name", "nama depan"]):
        base += ["first name", "contact first name"]

    if any(k in q for k in ["last name", "nama belakang", "surname"]):
        base += ["last name", "contact last name"]

    tokens = _dedup_keep_order(base)[:max_tokens]

    # query sangat pendek → tambahkan beberapa term umum
    if len(q.split()) <= 2:
        for t in ["amount", "price", "revenue", "month", "year"]:
            if t not in tokens and len(tokens) < max_tokens:
                tokens.append(t)

    probe_tail = " ".join(tokens).strip()
    return f"{question} {probe_tail}".strip()

def build_sparse_query_from_question(q: str, max_terms: int = 6) -> str:
    ql = (q or "").lower()
    terms: list[str] = []

    if "customer" in ql or "pelanggan" in ql:
        terms += ["customer", "customers"]
    if "product" in ql or "produk" in ql:
        terms += ["product", "products"]
    if "order" in ql or "pesanan" in ql:
        terms += ["order", "orders"]

    if any(w in ql for w in ["price", "list price", "expensive", "cheapest", "termahal", "termurah"]):
        terms += ["price", "list price", "msrp"]

    if "code" in ql:
        terms.append("code")
    if "name" in ql:
        terms.append("name")

    if not terms:
        toks = [t for t in re.split(r"\W+", ql) if t]
        stop = {"show","the","five","most","only","include","tampilkan","yang","paling","urut","top"}
        cand = [t for t in toks if t not in stop]
        terms = cand

    out = []
    seen = set()
    for t in terms:
        if t and t not in seen:
            seen.add(t)
            out.append(t)
        if len(out) >= max_terms:
            break

    return " ".join(out) or ql

def _doc_matches_sparse(md: Dict[str, Any], sparse_urns: list[str]) -> bool:
    if not sparse_urns or not md:
        return False
    urn = md.get("urn") or ""
    field_urn = md.get("field_urn") or ""
    sset = set(sparse_urns)
    if urn in sset:
        return True
    if field_urn and any(field_urn.startswith(u) for u in sset):
        return True
    return False

# =========================
# === INGEST (PURE CHROMA)
# =========================
def _ingest_one_json(col, meta: Dict[str, Any], source_name: str, debug_id: str):
    """
    Ingest satu baris json ke collection.
    Sekarang: tags_alias & tags_role ikut dimasukkan ke doc_text (ikut di-embed),
    dan kolom utama customerName diberi hint tambahan.
    """
    doc_id = meta.get("field_urn") or meta.get("urn")
    if not doc_id:
        doc_id = debug_id  # fallback

    # --------- PARSE ALIAS ----------
    raw_alias = meta.get("tags_alias") or meta.get("aliases") or meta.get("synonyms") or []
    if isinstance(raw_alias, str):
        try:
            parsed = json.loads(raw_alias)
            if isinstance(parsed, list):
                raw_alias = parsed
            else:
                raw_alias = [raw_alias]
        except Exception:
            raw_alias = [raw_alias]
    elif isinstance(raw_alias, list):
        raw_alias = raw_alias
    else:
        raw_alias = [str(raw_alias)] if raw_alias else []

    aliases_list = [str(a) for a in raw_alias]

    # --------- PARSE ROLE ----------
    raw_role = meta.get("tags_role") or []
    if isinstance(raw_role, str):
        try:
            parsed = json.loads(raw_role)
            if isinstance(parsed, list):
                raw_role = parsed
            else:
                raw_role = [raw_role]
        except Exception:
            raw_role = [raw_role]
    elif isinstance(raw_role, list):
        raw_role = raw_role
    else:
        raw_role = [str(raw_role)] if raw_role else []

    roles_list = [str(r) for r in raw_role]

    # simpan balik ke metadata sebagai JSON string
    meta["tags_alias"] = json.dumps(aliases_list, ensure_ascii=False)
    meta["tags_role"]  = json.dumps(roles_list, ensure_ascii=False)
    meta["_source"]    = source_name

    table = meta.get("table") or meta.get("tableName") or ""
    column = meta.get("column") or meta.get("columnName") or ""
    desc = meta.get("description") or ""

    alias_txt = ", ".join(aliases_list)
    role_txt  = ", ".join(roles_list)

    # hint ekstra kalau ini kolom utama nama customer
    extra_hint = ""
    lower_aliases = [a.lower() for a in aliases_list]
    joined_aliases = " ".join(lower_aliases)
    if (
        "customer name" in joined_aliases
        or "nama customer" in joined_aliases
        or "nama pelanggan" in joined_aliases
        or "account name" in joined_aliases
        or "company name" in joined_aliases
    ):
        extra_hint = "\nprimary_identity: main customer name column"

    doc_text = (
        f"table: {table}\n"
        f"column: {column}\n"
        f"description: {desc}\n"
        f"aliases: {alias_txt}\n"
        f"roles: {role_txt}"
        f"{extra_hint}"
    ).strip()

    existing = col.get(ids=[doc_id])
    exists = bool(existing and existing.get("ids"))

    if exists:
        existing_meta = (existing.get("metadatas") or [{}])[0]
        incoming_li = meta.get("last_ingested")
        existing_li = existing_meta.get("last_ingested")

        def to_dt(val):
            try:
                return datetime.fromisoformat(val)
            except Exception:
                return None

        should_update = False
        if incoming_li != existing_li:
            di = to_dt(incoming_li)
            de = to_dt(existing_li)
            if di and de:
                should_update = di > de
            else:
                should_update = True

        if should_update:
            col.update(
                ids=[doc_id],
                documents=[doc_text],
                metadatas=[meta],
            )
            return 0, 1, 0
        else:
            return 0, 0, 1
    else:
        col.add(
            ids=[doc_id],
            documents=[doc_text],
            metadatas=[meta],
        )
        return 1, 0, 0

def ingest_node(state: RAGState) -> RAGState:
    paths = state["jsonl_paths"]
    col = build_vector_store()

    before = col.count()
    added = updated = skipped = 0

    for path in paths:
        p = Path(path)
        if not p.exists():
            print(f"JSONL not found: {path}")
            continue

        print(f"[INGEST] reading {path}")
        for line_no, (raw, obj) in enumerate(read_jsonl(path), 1):
            if obj is None:
                try:
                    maybe = json.loads(raw)
                    if isinstance(maybe, list):
                        for i, item in enumerate(maybe):
                            if isinstance(item, dict):
                                _a, _u, _s = _ingest_one_json(
                                    col, item, p.name, f"{path}:{line_no}[{i}]"
                                )
                                added += _a
                                updated += _u
                                skipped += _s
                        continue
                except Exception:
                    pass
                print(f"[INGEST] skip {path}:{line_no} (not dict/list)")
                continue

            _a, _u, _s = _ingest_one_json(col, obj, p.name, f"{path}:{line_no}")
            added += _a
            updated += _u
            skipped += _s

    after = col.count()
    print("\n=== RAG Ingest Report ===")
    print(f"before : {before}")
    print(f"added  : {added}")
    print(f"updated: {updated}")
    print(f"skipped: {skipped}")
    print(f"after  : {after}")
    print("=========================\n")

    state["added"] = added
    state["updated"] = updated
    state["skipped"] = skipped
    return state

# =========================
# === RETRIEVE (PURE CHROMA)
# =========================
def retrieve_node(state: RAGState) -> RAGState:
    col = build_vector_store()
    q = (state["question"] or "").strip()

    # 1) Bangun query text (dense)
    query_text = build_dynamic_probe(q) if USE_DYNAMIC_PROBE else (q + " " + RETRIEVAL_PROBE).strip()

    # 2) Query ke Chroma + ambil distances
    res = col.query(
        query_texts=[query_text],
        n_results=max(3, TOP_K),
        include=["documents", "metadatas", "distances"],
    )

    docs      = (res.get("documents")  or [[]])[0]
    metas     = (res.get("metadatas")  or [[]])[0]
    dists     = (res.get("distances")  or [[]])[0]
    ids_list  = (res.get("ids")        or [[]])[0]

    # Safeguard panjang list
    n = min(
        len(docs),
        len(metas),
        len(dists) if dists else len(docs),
        len(ids_list) if ids_list else len(docs),
    )
    docs     = docs[:n]
    metas    = metas[:n]
    dists    = (dists or [0.0] * n)[:n]
    ids_list = (ids_list or [""] * n)[:n]

    # 3) Sparse dari DataHub
    sparse_query = build_sparse_query_from_question(q)
    sparse_urns  = datahub_sparse_search(sparse_query, count=10)

    # 4) Skor gabungan sederhana
    scored = []
    q_l = q.lower()

    for rank, (doc_text, md, dist, _id) in enumerate(zip(docs, metas, dists, ids_list)):
        base = -float(dist) if dist is not None else 0.0
        sparse_hit = _doc_matches_sparse(md, sparse_urns)

        col_name = (md.get("column") or md.get("columnName") or "").lower()

        # parse roles
        roles_raw = md.get("tags_role") or "[]"
        roles: List[str] = []
        if isinstance(roles_raw, str):
            try:
                parsed = json.loads(roles_raw)
                if isinstance(parsed, list):
                    roles = [str(r).lower() for r in parsed]
                else:
                    roles = [str(roles_raw).lower()]
            except Exception:
                roles = [str(roles_raw).lower()]
        elif isinstance(roles_raw, list):
            roles = [str(r).lower() for r in roles_raw]

        # parse aliases
        aliases_raw = md.get("tags_alias") or "[]"
        aliases: List[str] = []
        if isinstance(aliases_raw, str):
            try:
                parsed_a = json.loads(aliases_raw)
                if isinstance(parsed_a, list):
                    aliases = [str(a).lower() for a in parsed_a]
                else:
                    aliases = [str(aliases_raw).lower()]
            except Exception:
                aliases = [str(aliases_raw).lower()]
        elif isinstance(aliases_raw, list):
            aliases = [str(a).lower() for a in aliases_raw]

        # deteksi apakah ini kolom "customer name" utama
        is_main_customer_name = (
            "customername" in col_name
            or any(
                "customer name" in a
                or "nama customer" in a
                or "nama pelanggan" in a
                or "account name" in a
                or "company name" in a
                for a in aliases
            )
        )

        boosted = base + (SPARSE_BONUS if sparse_hit else 0.0)

        # bonus untuk query yang menyebut "name"/"nama"
        if any(k in q_l for k in ["name", "nama"]):
            # bonus umum untuk semua kolom yang berperan sebagai "name"
            if any(r == "name" or "name" in r for r in roles) or "name" in col_name:
                boosted += 0.3

            # bonus ekstra kalau query juga menyebut customer/pelanggan dan ini kolom utama nama customer
            if any(k in q_l for k in ["customer", "customers", "pelanggan"]) and is_main_customer_name:
                boosted += 0.4

        scored.append({
            "rank": rank,
            "id": _id,
            "doc": doc_text,
            "meta": md or {},
            "dist": float(dist) if dist is not None else 0.0,
            "base": base,
            "sparse": sparse_hit,
            "boosted": boosted,
        })

    # 5) Rerank
    reranked = sorted(scored, key=lambda x: x["boosted"], reverse=True)

    # 6) LOGS
    print("\n===== RETRIEVE DEBUG (PURE CHROMA) =====")
    print(f"Question         : {q}")
    print(f"Query (dense)    : {query_text}")
    print(f"Dense docs (raw) : {len(scored)}")
    print(f"Sparse query     : {sparse_query!r}")
    if sparse_urns:
        print("Sparse URNs      :")
        for u in sparse_urns:
            print(f"  - {u}")
    else:
        print("Sparse URNs      : (none)")

    print("\nDense (before rerank):")
    for item in scored:
        md   = item["meta"]
        urn  = md.get("urn", "")
        tbl  = md.get("table") or md.get("tableName") or ""
        col  = md.get("column") or md.get("columnName") or ""
        print(
            f"  • rank={item['rank']}  dist={item['dist']:.4f}  base={item['base']:.4f}  "
            f"sparse={item['sparse']}  -> id={item['id']}  urn={urn}  table={tbl}  column={col}"
        )

    print("\nAfter rerank (dense + sparse + name bonus):")
    for i, item in enumerate(reranked):
        md   = item["meta"]
        urn  = md.get("urn", "")
        tbl  = md.get("table") or md.get("tableName") or ""
        col  = md.get("column") or md.get("columnName") or ""
        print(
            f"  • final_rank={i}  boosted={item['boosted']:.4f}  "
            f"(from_dense_rank={item['rank']})  sparse={item['sparse']}  "
            f"-> id={item['id']}  urn={urn}  table={tbl}  column={col}"
        )
    print("===== END RETRIEVE DEBUG =====\n")

    # 7) Build context
    allowed   = defaultdict(list)
    alias_map = defaultdict(dict)

    for item in reranked:
        md = item["meta"] or {}
        t  = (md.get("table") or md.get("tableName") or "").strip()
        c  = (md.get("column") or md.get("columnName") or "").strip()
        if t and c:
            allowed[t].append(c)

            aliases = md.get("aliases") or md.get("tags_alias") or md.get("synonyms") or []
            if isinstance(aliases, str):
                try:
                    maybe = json.loads(aliases)
                    if isinstance(maybe, list):
                        aliases = maybe
                    else:
                        aliases = [aliases]
                except Exception:
                    aliases = [aliases]
            for al in aliases:
                al = str(al).strip()
                if al:
                    alias_map[t][al.lower()] = c

    q_l = q.lower()
    likely_tables = set()
    if any(w in q_l for w in ["customer", "customers", "pelanggan"]):
        likely_tables.add("customers")
    if any(w in q_l for w in ["order", "orders", "pesanan"]):
        likely_tables.add("orders")
    if any(w in q_l for w in ["product", "products", "produk"]):
        likely_tables.add("products")

    if likely_tables:
        allowed = {t: cols for t, cols in allowed.items() if t.lower() in likely_tables}
    else:
        allowed = dict(allowed)

    allowed_lines = []
    for t, cols in allowed.items():
        uniq = sorted(set(cols))
        allowed_lines.append(f"- {t}: {', '.join(uniq)}")
    allowed_block = "### ALLOWED IDENTIFIERS\n" + ("\n".join(allowed_lines) if allowed_lines else "- (none)")

    cards_text = "\n".join([it["doc"] for it in reranked])

    sparse_info = ""
    if sparse_urns:
        sparse_info = "\n\n### SPARSE DATASETS (from DataHub)\n" + "\n".join(f"- {u}" for u in sparse_urns)

    context = (allowed_block + sparse_info + "\n\n" + cards_text).strip()
    context += f"\n\n### _INTERNAL_HINT\n{json.dumps({'alias_map': alias_map}, ensure_ascii=False)}"

    state["context"] = context
    return state

# =========================
# === INTENT / RRF ========
# =========================
def intent_terms_from_question(q: str) -> list[str]:
    q = q.lower()
    terms: list[str] = []

    if "name" in q or "nama" in q:
        terms += ["name", "customername", "contactfirstname", "contactlastname", "productname"]

    if "price" in q or "harga" in q or "expensive" in q or "cheapest" in q:
        terms += ["price", "msrp", "priceeach", "buyprice", "listprice"]

    if "date" in q or "tanggal" in q or "period" in q:
        terms += ["date", "orderdate", "paymentdate", "shippeddate"]

    if "id" in q or "code" in q or "kode" in q:
        terms += ["id", "code", "customernumber", "productcode", "ordernumber"]

    return list(dict.fromkeys(terms))

def rrf(rank: int, k: int = RRF_K) -> float:
    return 1.0 / (k + rank)

# =========================
# === SQL POSTPROCESS =====
# =========================
_JOIN_KW = r"(?:inner\s+join|left\s+join|right\s+join|full\s+join|join)"

def _has_join(sql: str) -> bool:
    return re.search(rf"\b{_JOIN_KW}\b", sql, flags=re.I) is not None

def _should_use_backtick(identifier: str) -> bool:
    if not identifier: return False
    clean = identifier.strip('`"')
    if not clean: return False
    first_char = clean[0]
    if first_char.isupper() or first_char.isdigit():
        return True
    if not re.match(r'^[a-z_][a-z0-9_]*$', clean, re.I):
        return True
    return False

def _smart_backtick(identifier: str) -> str:
    clean = identifier.strip('`"')
    return f"`{clean}`" if _should_use_backtick(clean) else clean

def _split_qualified(name: str) -> List[str]:
    return [p.strip('`" ') for p in name.split(".") if p.strip("`\" ")]

def _fix_bad_backticks(sql: str) -> str:
    def repl_fqn(m: re.Match) -> str:
        inside = m.group(1)
        if "." not in inside:
            return _smart_backtick(inside)
        parts = _split_qualified(inside)
        if len(parts) == 0:
            return inside
        return ".".join(_smart_backtick(p) for p in parts)
    sql = re.sub(r"`([^`]+?\.[^`]+?)`", repl_fqn, sql)
    sql = re.sub(r"(\b[A-Za-z_][\w]*)\.`([^`]+?)`",
                 lambda m: f"{_smart_backtick(m.group(1))}."
                           + ".".join(_smart_backtick(p) for p in _split_qualified(m.group(2))), sql)
    sql = re.sub(r"`([^`]+?)`\.(\b[A-Za-z_][\w]*)",
                 lambda m: ".".join(_smart_backtick(p) for p in _split_qualified(m.group(1)))
                           + f".{_smart_backtick(m.group(2))}", sql)
    sql = re.sub(r'`([^`]+?)`(?!\s*\.)', lambda m: _smart_backtick(m.group(1)), sql, flags=re.I)
    return sql

def _explicit_limit_from_question(question: str) -> Optional[int]:
    q = (question or "").lower()
    m = re.search(r"\blimit\s+(\d{1,4})\b", q)
    if m: return max(1, min(int(m.group(1)), 1_000_000))
    m = re.search(r"\btop\s+(\d{1,4})\b", q)
    if m: return max(1, min(int(m.group(1)), 1_000_000))
    m = re.search(r"\b(show|first|tampilkan|ambil)\s+(\d{1,4})\b", q)
    if m: return max(1, min(int(m.group(2)), 1_000_000))
    return None

def _maybe_add_limit(question: str, sql: str) -> str:
    if re.search(r"\blimit\s+\d+\b", sql, flags=re.I):
        return sql
    n = _explicit_limit_from_question(question)
    if n is None:
        return sql
    s = sql.rstrip().rstrip(";")
    return s + f" LIMIT {n};"

def _wants_distinct_from_question(question: str) -> bool:
    q = (question or "").lower()
    return any(k in q for k in [
        "distinct", "unique", "no duplicate", "tanpa duplikat",
        "unik", "dedup", "de-dup", "hanya yang unik"
    ])

def _wants_all_rows_from_question(question: str) -> bool:
    q = (question or "").lower()
    return any(k in q for k in [
        "all rows", "include duplicates", "dengan duplikat",
        "raw rows", "semua baris", "tanpa distinct", "jangan distinct"
    ])

def _has_distinct(sql: str) -> bool:
    return bool(re.search(r"^\s*select\s+distinct\b", sql, flags=re.I))

def _add_distinct(sql: str) -> str:
    if _has_distinct(sql):
        return sql
    return re.sub(r"^\s*select\s+", "SELECT DISTINCT ", sql, flags=re.I)

def _remove_distinct(sql: str) -> str:
    return re.sub(r"^\s*select\s+distinct\s+", "SELECT ", sql, flags=re.I)

def _ensure_reasonable_distinct(sql: str, question: str) -> str:
    if not sql:
        return sql
    if _wants_distinct_from_question(question):
        return _add_distinct(sql)
    if _wants_all_rows_from_question(question):
        return _remove_distinct(sql)
    return _remove_distinct(sql)

def _fix_missing_space_before_clauses(sql: str) -> str:
    sql = re.sub(r"\s+(?=(ORDER\s+BY|GROUP\s+BY|HAVING|LIMIT|WHERE)\b)", " ", sql, flags=re.I)
    sql = re.sub(
        r"([)\],;`])(?=(ORDER\s+BY|GROUP\s+BY|HAVING|LIMIT|WHERE)\b)",
        r"\1 ",
        sql,
        flags=re.I,
    )
    sql = re.sub(r"([A-Za-z0-9_\.])(?=(LIMIT|WHERE|HAVING)\b)", r"\1 ", sql)
    sql = re.sub(r"([A-Za-z0-9_\.])(?=ORDER\s+BY\b)", r"\1 ", sql)
    sql = re.sub(r"([A-Za-z0-9_\.])(?=GROUP\s+BY\b)", r"\1 ", sql)
    sql = re.sub(r"\s+(ORDER\s+BY|GROUP\s+BY|HAVING|LIMIT|WHERE)\b", r" \1", sql, flags=re.I)
    return sql

def _collapse_spaces_outside_quotes(s: str) -> str:
    parts = re.split(r"('(?:''|[^'])*')", s)
    for i in range(len(parts)):
        if i % 2 == 0:
            parts[i] = re.sub(r"[ \t]{2,}", " ", parts[i])
    return "".join(parts)

def _extract_order_by_clause(sql: str) -> Optional[re.Match]:
    return re.search(r"\border\s+by\s+(.*?)(?=(\blimit\b|;|$))", sql, flags=re.I | re.S)

def _remove_order_by_clause(sql: str) -> str:
    return re.sub(r"\s+order\s+by\s+.*?(?=(\blimit\b|;|$))", "", sql, flags=re.I | re.S)

def _question_requests_ordering(question: str) -> bool:
    q = (question or "").lower()
    keywords = [
        "order by", "sort", "sorted", "ascending", "descending", "asc", "desc",
        "alphabetical", "alphabetically", "latest", "newest", "oldest",
        "highest", "lowest", "top", "bottom", "most expensive", "cheapest",
        "urut", "urutkan", "diurutkan", "menaik", "menurun",
        "terbesar", "terkecil", "tertinggi", "terendah", "terbaru", "terlama",
        "termahal", "termurah",
    ]
    return any(k in q for k in keywords)

def _postprocess_sql(sql: str, question: str, context: str = "") -> str:
    if not sql:
        return sql
    s = sql.strip()

    def _norm_kw(text: str) -> str:
        CHUNKS = re.compile(
            r"(`[^`]*`|\"([^\"\\]|\\.)*\"|'([^'\\]|\\.)*')",
            re.DOTALL,
        )
        KEYWORDS = [
            "select","from","where","group by","order by","having","limit",
            "inner join","left join","right join","full join","join","on",
        ]

        def normalize_code_segment(seg: str) -> str:
            for kw in KEYWORDS:
                pattern = rf"(?<![A-Za-z0-9_]){kw}(?![A-Za-z0-9_])"
                seg = re.sub(pattern, lambda m: m.group(0).upper(), seg, flags=re.I)
            seg = re.sub(r"[ \t]+", " ", seg)
            return seg

        out, i = [], 0
        for m in CHUNKS.finditer(text):
            out.append(normalize_code_segment(text[i:m.start()]))
            out.append(m.group(0))
            i = m.end()
        out.append(normalize_code_segment(text[i:]))

        return "".join(out)

    s = _norm_kw(s)
    if not s.endswith(";"):
        s += ";"

    s = _fix_bad_backticks(s)
    s = _ensure_reasonable_distinct(s, question)

    ob_match = _extract_order_by_clause(s)
    if ob_match and not _question_requests_ordering(question):
        s = _remove_order_by_clause(s)

    s = _maybe_add_limit(question, s)
    s = _fix_missing_space_before_clauses(s)
    s = _collapse_spaces_outside_quotes(s)

    return s

# ---------- Whitelist parsing + alias repair ----------
_ALNUM = re.compile(r"[A-Za-z_][A-Za-z0-9_]*")

def _extract_internal_hint(ctx: str) -> Dict[str, Dict[str, str]]:
    m = re.search(r"### _INTERNAL_HINT\s+({.*})\s*$", ctx, flags=re.S)
    if not m:
        return {}
    try:
        obj = json.loads(m.group(1))
        return obj.get("alias_map") or {}
    except Exception:
        return {}

def _parse_allowed_identifiers(ctx: str) -> Dict[str, set]:
    tables: Dict[str, set] = {}
    m = re.search(r"### ALLOWED IDENTIFIERS(.*?)(?:\n### |\Z)", ctx, flags=re.S)
    if not m:
        return tables
    block = m.group(1)
    for line in block.splitlines():
        line = line.strip()
        if not line.startswith("- "):
            continue
        parts = line[2:].split(":", 1)
        if len(parts) != 2:
            continue
        table = parts[0].strip().strip("`")
        cols = [c.strip().strip("`") for c in parts[1].split(",") if c.strip()]
        if table:
            tables.setdefault(table, set()).update(cols)
    return tables

def _identify_used_identifiers(sql: str) -> Dict[Optional[str], set]:
    qualified = re.findall(r"`?([A-Za-z_][A-Za-z0-9_]*)`?\s*\.\s*`?([A-Za-z_][A-Za-z0-9_]*)`?", sql)
    out: Dict[Optional[str], set] = {}
    for t, c in qualified:
        out.setdefault(t, set()).add(c)
    keywords = {"select","from","where","group","by","order","limit","join","left","right","inner","on","and","or","not","as","distinct","count","sum","avg","min","max","having","asc","desc","like","in","between","is","null","case","when","then","else","end"}
    for tok in re.findall(r"\b([A-Za-z_][A-Za-z0-9_]*)\b", sql):
        if tok.lower() in keywords:
            continue
        out.setdefault(None, set()).add(tok)
    return out

def _validate_identifiers(sql: str, ctx: str) -> List[str]:
    allowed = _parse_allowed_identifiers(ctx)
    used = _identify_used_identifiers(sql)
    errs: List[str] = []
    for tbl, cols in used.items():
        if tbl is None:
            continue
        if tbl not in allowed:
            errs.append(f"table `{tbl}` not in ALLOWED")
            continue
        bad = [c for c in cols if c not in allowed[tbl]]
        if bad:
            errs.append(f"{tbl} -> invalid columns: {', '.join(bad)}")
    return errs

def _repair_with_aliases(sql: str, ctx: str) -> str:
    alias_map_per_table = _extract_internal_hint(ctx)

    classic_fallback = {
        "products": {
            "name": "productName",
            "productname": "productName",
            "listprice": "MSRP",
        },
        "orderdetails": {
            "listprice": "priceEach",
            "price": "priceEach",
        },
        "customers": {
            "name": "customerName",
        },
    }

    global_map: Dict[str, Optional[str]] = {}
    for _t, amap in (alias_map_per_table or {}).items():
        for al, canon in (amap or {}).items():
            al = str(al).lower()
            if al not in global_map:
                global_map[al] = canon
            elif global_map[al] != canon:
                global_map[al] = None

    for _t, amap in classic_fallback.items():
        for al, canon in amap.items():
            if al not in global_map:
                global_map[al] = canon
            elif global_map[al] is None:
                continue

    keywords = {
        "select","from","where","group","by","order","limit","join","left","right","inner",
        "on","and","or","not","as","distinct","count","sum","avg","min","max","having",
        "asc","desc","like","in","between","is","null","case","when","then","else","end"
    }
    pattern = re.compile(r"\b([A-Za-z_][A-Za-z0-9_]*)\b")

    def smart_repl(m):
        w = m.group(0)
        lw = w.lower()
        if lw in keywords:
            return w
        canon = global_map.get(lw)
        return canon if canon and _ALNUM.match(canon) else w

    return pattern.sub(smart_repl, sql)

# ---------- Prompt ----------
def _make_generation_prompt(ctx: str, question: str) -> str:
    return f"""
You are a senior SQL assistant for MySQL 8.0.

Return a runnable query using ONLY tables/columns listed in the ALLOWED IDENTIFIERS section of Context.
If a needed column name appears as an alias in Context, use its canonical column name.

########################
# STRICT OUTPUT FORMAT #
########################
{{ "sql": "<final MySQL query>", "notes": "<brief rationale; for EACH JOIN, name the exact selected/grouped/filtered column it contributes>" }}
# No extra keys. No Markdown. End SQL with a semicolon.
# If the final SQL has NO JOINs, set "notes" exactly to: "No joins used."

##################
# SQL CONSTRAINT #
##################
- MySQL 8.0; backticks only when needed; ANSI joins.

########################################
# ANCHOR TABLE & MINIMAL-JOIN POLICY   #
########################################
- Pick as ANCHOR the table containing the most required columns.
- Only JOIN if you SELECT, GROUP BY, or WHERE on a column that is NOT in the ANCHOR.
- After drafting, AUDIT each JOIN: if you cannot state which output/filter column comes from that table, REMOVE it.

########################################
# DISTINCT & GROUP BY
########################################
- Avoid DISTINCT unless explicitly asked or needed to deduplicate.

################
# CONTEXT
################
{ctx}

Question:
{question}
""".strip()

def _sanitize_notes(sql: str, notes: str) -> str:
    if not notes:
        return notes
    if not _has_join(sql or ""):
        return "No joins used."
    return notes.strip()

# ---------- Generation (with validation & regeneration) ----------
def generate_node(state: RAGState) -> RAGState:
    llm = ChatOllama(
        model=CHAT_MODEL,
        base_url=OLLAMA_BASE_URL,
        temperature=0.0,
        seed=42
    )
    ctx = state.get("context", "")
    question = state.get("question", "")
    prompt = _make_generation_prompt(ctx, question)
    resp = llm.invoke(prompt)
    content = getattr(resp, "content", None) or str(resp)

    answer_out = content
    draft_sql = ""
    final_sql = ""
    try:
        obj = json.loads(content)
        draft_sql = (obj.get("sql") or "").strip()
        notes = (obj.get("notes") or "").strip()

        if draft_sql:
            _dbg_print_two("RAG SQL Code :", draft_sql)

        final_sql = _postprocess_sql(draft_sql, question, ctx) if draft_sql else draft_sql
        if final_sql:
            final_sql = _repair_with_aliases(final_sql, ctx)
            final_sql = _postprocess_sql(final_sql, question, ctx)

        violations = _validate_identifiers(final_sql or "", ctx)
        if violations:
            force_msg = (
                "The previous SQL used identifiers not present in ALLOWED IDENTIFIERS:\n- "
                + "\n- ".join(violations)
                + "\nRewrite the SQL using ONLY identifiers listed in ALLOWED IDENTIFIERS. "
                  "If a synonym appears, choose the canonical column."
            )
            strict_prompt = _make_generation_prompt(ctx + "\n\n" + force_msg, question)
            resp2 = llm.invoke(strict_prompt)
            content2 = getattr(resp2, "content", None) or str(resp2)
            try:
                obj2 = json.loads(content2)
                draft2 = (obj2.get("sql") or "").strip()
                notes2 = (obj2.get("notes") or "").strip()
            except Exception:
                draft2, notes2 = content2, ""
            sql2 = _postprocess_sql(draft2 or "", question, ctx) if draft2 else ""
            if sql2:
                sql2 = _repair_with_aliases(sql2, ctx)
                sql2 = _postprocess_sql(sql2, question, ctx)
            v1 = len(violations)
            v2 = len(_validate_identifiers(sql2 or "", ctx))
            if v2 <= v1 and sql2:
                final_sql, notes = sql2, (notes2 or notes)

        if final_sql:
            _dbg_print_two("Sanitize SQL Code :", final_sql)

        notes = _sanitize_notes(final_sql or "", notes or "")

        if final_sql:
            answer_out = final_sql + ("\n\n-- notes: " + notes if notes else "")
        else:
            answer_out = (notes or "").strip() or content
    except Exception:
        pass

    state["answer"] = answer_out
    state["draft_sql"] = draft_sql
    state["final_sql"] = final_sql
    return state

# =========================
# === APP GRAPH ===========
# =========================
def build_app():
    g = StateGraph(RAGState)
    g.add_node("ingest", ingest_node)
    g.add_node("retrieve", retrieve_node)
    g.add_node("generate", generate_node)
    g.set_entry_point("ingest")
    g.add_edge("ingest", "retrieve")
    g.add_edge("retrieve", "generate")
    g.add_edge("generate", END)
    return g.compile()

# =========================
# === MCP UTIL ============
# =========================
async def _mcp_call(client: Client, tool: str, payload: dict | None = None, timeout: int = MCP_TIMEOUT):
    coro = client.call_tool(tool, payload or {})
    return await asyncio.wait_for(coro, timeout=timeout)

def _extract_payload(result) -> Any:
    if hasattr(result, 'data') and result.data is not None:
        return result.data
    if hasattr(result, "structuredContent") and result.structuredContent is not None:
        return result.structuredContent
    if hasattr(result, "content") and result.content:
        return result.content[0].text if isinstance(result.content, list) else str(result.content)
    return result

def _ensure_json(d: Any) -> Optional[dict]:
    if isinstance(d, dict):
        return d
    if isinstance(d, str):
        try:
            return json.loads(d)
        except Exception:
            return None
    return None

def display_tables_list(data: dict):
    tables = data.get("tables", [])
    total = data.get("total_tables", len(tables))
    if not tables:
        print("No tables found.")
        return
    print("\nAvailable tables:")
    for i, t in enumerate(tables, start=1):
        print(f"  {i:>2}. {t}")
    print(f"\nTotal: {total} table(s)")

def display_table_from_json(data):
    preview_rows = data.get("preview_rows", [])
    preview_cols = data.get("preview_columns", [])
    total_rows = data.get("total_rows", len(preview_rows))
    total_cols = data.get("total_columns", len(preview_cols))
    if preview_rows and preview_cols:
        df = pd.DataFrame(preview_rows, columns=preview_cols)
        df.index = df.index + 1
        if total_rows > len(preview_rows):
            df.loc["..."] = ["..."] * len(df.columns)
        if total_cols > len(df.columns):
            df["..."] = "..."
        print("\nQuery Result (preview):")
        print(df.to_string())
        print(f"\nShowing preview: {len(preview_rows)} rows × {len(preview_cols)} columns "
              f"(full result: {total_rows} rows × {total_cols} columns)")
    else:
        print("Query executed but returned no rows.")

# =========================
# === REPL RUNNER =========
# =========================
async def repl(paths: List[str]):
    app = build_app()

    transport = StreamableHttpTransport(MCP_BASE)
    async with Client(transport=transport) as mcp:
        try:
            tools = await mcp.list_tools()
            names = [getattr(t, "name", None) for t in tools]
            print("\nMCP tools:", ", ".join([n for n in names if n]) or "(none)")
        except Exception as e:
            print(f"Could not list MCP tools: {e}")

        first_q = input("Enter a test question (or press Enter to skip): ").strip()
        if not first_q:
            first_q = "Which columns describe customer identity?"

        init: RAGState = {
            "jsonl_paths": paths,
            "question": first_q,
            "added": 0,
            "updated": 0,
            "skipped": 0,
            "context": "",
            "answer": "",
            "draft_sql": "",
            "final_sql": "",
        }

        out = app.invoke(init)

        col = build_vector_store()
        total = col.count()
        print("\n=== RAG Ingest Report ===")
        print(f"Added (first run): {out.get('added', 0)}")
        print(f"Updated (upserted): {out.get('updated', 0)}")
        print(f"Skipped (unchanged): {out.get('skipped', 0)}")
        print(f"Total docs now in collection: {total}")
        print(f"Chroma path: {Path(CHROMA_DIR).resolve()} | Collection: {COLLECTION_NAME}")

        print("\n--- Generated Answer (RAG) ---\n")
        generated = out.get("answer", "(no answer)")
        print(generated)

        async def exec_via_mcp(sql_block: str, final_sql: Optional[str] = None):
            sql_only = sql_block.split("\n\n-- notes:")[0].strip()
            sql_clean = (final_sql or sql_only).rstrip(";").strip()

            try:
                lt = await _mcp_call(mcp, "list_tables", {})
                lt_payload = _ensure_json(_extract_payload(lt)) or {}
                if lt_payload.get("success"):
                    display_tables_list(lt_payload)
            except Exception as e:
                print(f"list_tables failed (skip): {e}")

            print(f"\nSQL (RAG -> MCP):\n{sql_clean};")

            try:
                res = await _mcp_call(mcp, "query_db", {"sql": sql_clean})
                payload = _ensure_json(_extract_payload(res))
                if isinstance(payload, dict) and payload.get("success"):
                    display_table_from_json(payload)
                    return payload
                else:
                    err = payload.get("error") if isinstance(payload, dict) else str(payload)
                    print(f"query_db failed: {err}")
                    return None
            except asyncio.TimeoutError:
                print("MCP query_db timeout")
                return None
            except Exception as e:
                print(f"MCP query_db error: {e}")
                return None

        await exec_via_mcp(generated, out.get("final_sql"))

        print("\nAsk me more questions! (type 'exit' to quit)")
        while True:
            q = input("\nQuestion> ").strip()
            if not q or q.lower() in {"exit", "quit", "q"}:
                print("Bye.")
                break

            state: RAGState = {
                "jsonl_paths": paths,
                "question": q,
                "added": 0,
                "updated": 0,
                "skipped": 0,
                "context": "",
                "answer": "",
                "draft_sql": "",
                "final_sql": "",
            }
            state = retrieve_node(state)
            state = generate_node(state)

            print("\n--- Generated Answer (RAG) ---\n")
            print(state.get("answer", "(no answer)"))

            await exec_via_mcp(state.get("answer", ""), state.get("final_sql"))

def main():
    paths = resolve_jsonl_paths(JSON_INPUT)
    if not paths:
        raise RuntimeError(
            "No JSONL inputs found.\n"
            "Set EXPORT_JSONL_OUTPUT to one of:\n"
            "  • a single file: ./exports/classicmodels.jsonl\n"
            "  • a directory:   ./datahub_metadata\n"
            "  • a glob:        ./datahub_metadata/*.jsonl\n"
        )
    print("JSONL inputs:")
    for p in paths:
        print(" -", p)

    try:
        asyncio.run(repl(paths))
    except RuntimeError as e:
        if "asyncio.run() cannot be called" in str(e):
            import nest_asyncio
            nest_asyncio.apply()
            loop = asyncio.get_event_loop()
            loop.run_until_complete(repl(paths))
        else:
            raise

if __name__ == "__main__":
    main()

JSONL inputs:
 - metadata_jsonl/classicmodels.jsonl

MCP tools: list_tables, query_db, query_db_full


Enter a test question (or press Enter to skip):  Show distinct customer countries.


[INGEST] reading metadata_jsonl/classicmodels.jsonl

=== RAG Ingest Report ===
before : 59
added  : 0
updated: 0
skipped: 59
after  : 59


===== RETRIEVE DEBUG (PURE CHROMA) =====
Question         : Show distinct customer countries.
Query (dense)    : Show distinct customer countries. id date status count qty quantity number customer customer name
Dense docs (raw) : 5
Sparse query     : 'customer customers'
Sparse URNs      :
  - urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.customers,PROD)
  - urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.payments,PROD)
  - urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.orders,PROD)

Dense (before rerank):
  • rank=0  dist=0.8166  base=-0.8166  sparse=True  -> id=urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.customers,PROD)::country  urn=urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.customers,PROD)  table=customers  column=country
  • rank=1  dist=1.0013  base=-1.0013  sparse=True  -> id=urn:li:dataset:


Question>  EXIT


Bye.


## **CHECK COLLECTION**

In [ ]:
import chromadb
from pprint import pprint

CHROMA_DIR = "./chroma_db"
COLLECTION_NAME = "classicmodels"

client = chromadb.PersistentClient(path=CHROMA_DIR)
col = client.get_collection(COLLECTION_NAME)

print("== BASIC INFO ==")
print("name :", col.name)
print("count:", col.count())

# ambil beberapa dokumen pertama
print("\n== FIRST 5 DOCS ==")
res = col.get(limit=5)  # bisa pakai where={} juga
pprint(res)

## **DELETE COLLECTION**

In [ ]:
import chromadb

CHROMA_DIR = "./chroma_db"
COLLECTION_NAME = "classicmodels"

client = chromadb.PersistentClient(path=CHROMA_DIR)

# cek dulu ada atau nggak
try:
    client.get_collection(COLLECTION_NAME)  # kalau gak ada bakal raise
    client.delete_collection(COLLECTION_NAME)
    print(f"[OK] collection '{COLLECTION_NAME}' deleted.")
except Exception as e:
    print(f"[INFO] collection '{COLLECTION_NAME}' not found or already deleted. ({e})")